# ETL — Dataset predictivo de mantenimiento

Construye un dataset supervisado a partir de las tablas crudas exportadas por `php artisan predictive:export-raw`.

**Unidad de análisis:** un activo en una fecha de corte mensual.

**Objetivo:** `falla_30d` indica si el activo tuvo al menos un ticket correctivo en los 30 días posteriores al corte. Todas las predictoras usan solo información disponible hasta el corte para evitar fuga de información.

**Salida:** `data/processed/predictive_maintenance_dataset.csv`.

In [1]:

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "reports" / "figures"
for directory in (PROCESSED, FIGURES):
    directory.mkdir(parents=True, exist_ok=True)

DEMO_TENANTS = [
    "acero-norte",
    "alimentos-valle",
    "logistica-pacifico",
    "textiles-andinos",
]

HISTORY_START = pd.Timestamp("2023-01-01")
HISTORY_END = pd.Timestamp("2026-01-31 23:59:59")
CUTOFFS = pd.date_range("2024-02-01", periods=24, freq="MS")
FEATURE_WINDOW_DAYS = 180
TARGET_WINDOW_DAYS = 30

print(f"Directorio de trabajo: {ROOT}")
print(f"Cortes: {CUTOFFS.min().date()} .. {CUTOFFS.max().date()} ({len(CUTOFFS)} cortes)")
print(f"Ventana de predictores: {FEATURE_WINDOW_DAYS} días")
print(f"Ventana objetivo: {TARGET_WINDOW_DAYS} días")


Directorio de trabajo: /workspace
Cortes: 2024-02-01 .. 2026-01-01 (24 cortes)
Ventana de predictores: 180 días
Ventana objetivo: 30 días


## 1. Carga de tablas crudas

Se filtran las cuatro empresas demo. La empresa existente `elyon-yireh` no forma parte del dataset.

In [2]:

assets_raw = pd.read_csv(
    RAW / "asset_cards.csv",
    parse_dates=["created_at", "updated_at"],
    low_memory=False,
)
tickets = pd.read_csv(
    RAW / "tickets.csv",
    parse_dates=["created_at", "categorized_at"],
    low_memory=False,
)
closures = pd.read_csv(
    RAW / "ticket_closures.csv",
    parse_dates=["closed_at"],
    low_memory=False,
)
executions = pd.read_csv(
    RAW / "preventive_executions.csv",
    parse_dates=[
        "scheduled_at",
        "original_scheduled_at",
        "started_at",
        "completed_at",
    ],
    low_memory=False,
)
categories = pd.read_csv(RAW / "maintenance_categories.csv", low_memory=False)
locations = pd.read_csv(RAW / "locations.csv", low_memory=False)

assets_raw = assets_raw[assets_raw["tenant_id"].isin(DEMO_TENANTS)].copy()
tickets = tickets[tickets["tenant_id"].isin(DEMO_TENANTS)].copy()
closures = closures[closures["tenant_id"].isin(DEMO_TENANTS)].copy()
executions = executions[executions["tenant_id"].isin(DEMO_TENANTS)].copy()
categories = categories[categories["tenant_id"].isin(DEMO_TENANTS)].copy()

print("Filas crudas:")
print(f"  asset_cards: {len(assets_raw):,}")
print(f"  tickets: {len(tickets):,}")
print(f"  ticket_closures: {len(closures):,}")
print(f"  preventive_executions: {len(executions):,}")


Filas crudas:
  asset_cards: 500
  tickets: 4,179
  ticket_closures: 4,136
  preventive_executions: 11,528


## 2. Normalización de activos

Los campos operativos viven dentro del JSON dinámico de `asset_cards.data`.

In [3]:

ASSET_FIELDS = {
    "nombre",
    "tipo_activo",
    "sede",
    "fecha_instalacion",
    "criticidad",
}


def extract_asset_fields(raw_data: object) -> dict:
    if not isinstance(raw_data, str) or raw_data.strip() == "":
        return {}

    try:
        parsed = json.loads(raw_data)
    except json.JSONDecodeError:
        return {}

    values = {}
    for section in parsed.values():
        if not isinstance(section, dict):
            continue
        for field_name, payload in section.items():
            if field_name not in ASSET_FIELDS:
                continue
            if isinstance(payload, dict):
                values[field_name] = payload.get("value")
            else:
                values[field_name] = payload
    return values


asset_fields = assets_raw["data"].apply(extract_asset_fields).apply(pd.Series)
assets = pd.concat(
    [
        assets_raw[["tenant_id", "id", "code", "created_at"]].reset_index(drop=True),
        asset_fields.reset_index(drop=True),
    ],
    axis=1,
)
assets = assets.rename(columns={"id": "asset_id", "code": "codigo_activo"})
assets["fecha_instalacion"] = pd.to_datetime(assets["fecha_instalacion"], errors="coerce")
assets["installed_at"] = assets["fecha_instalacion"].fillna(assets["created_at"])
assets["installed_at"] = pd.to_datetime(assets["installed_at"], errors="coerce").fillna(HISTORY_START)
assets["tipo_activo"] = assets["tipo_activo"].fillna("Desconocido").astype(str)
assets["sede"] = assets["sede"].fillna("Desconocida").astype(str)
assets["criticidad"] = assets["criticidad"].fillna("medium").astype(str)
assets = assets[
    [
        "tenant_id",
        "asset_id",
        "codigo_activo",
        "tipo_activo",
        "sede",
        "criticidad",
        "installed_at",
    ]
]

print(f"Activos demo: {len(assets):,}")
print(assets.groupby("tenant_id").size().to_string())
assets.head()


Activos demo: 500
tenant_id
acero-norte           125
alimentos-valle       125
logistica-pacifico    125
textiles-andinos      125


,tenant_id,asset_id,codigo_activo,tipo_activo,sede,criticidad,installed_at
0,acero-norte,1,DEMO-0001,Compresor,Planta norte,critical,2013-01-26
1,acero-norte,2,DEMO-0002,Prensa hidráulica,Planta norte,high,2017-06-20
2,acero-norte,3,DEMO-0003,Puente grúa,Centro de servicios,high,2021-09-28
3,acero-norte,4,DEMO-0004,Bomba industrial,Planta norte,medium,2015-06-20
4,acero-norte,5,DEMO-0005,Motor eléctrico,Planta principal,critical,2019-10-31


## 3. Preparación de eventos

Se unen tickets con cierres y se indexan por tenant + activo para que los IDs repetidos entre tenants no se mezclen.

In [4]:

tickets = tickets.dropna(subset=["created_at"]).copy()
tickets = tickets[
    (tickets["created_at"] >= HISTORY_START)
    & (tickets["created_at"] <= HISTORY_END + pd.Timedelta(days=TARGET_WINDOW_DAYS))
].copy()
tickets["is_corrective"] = (
    tickets["maintenance_type"].fillna("").astype(str).str.lower().eq("corrective")
)

closures = closures.dropna(subset=["closed_at"]).copy()
tickets = tickets.merge(
    closures[["tenant_id", "ticket_id", "closed_at"]],
    left_on=["tenant_id", "id"],
    right_on=["tenant_id", "ticket_id"],
    how="left",
    validate="many_to_one",
)
tickets["resolution_hours"] = (
    tickets["closed_at"] - tickets["created_at"]
).dt.total_seconds() / 3600

priority_order = {"low": 1, "medium": 2, "high": 3, "critical": 4}
tickets["priority_num"] = tickets["priority"].map(priority_order)

category_map = {
    (row.tenant_id, int(row.id)): row.name
    for row in categories.itertuples(index=False)
}
tickets["categoria"] = [
    category_map.get((tenant_id, int(category_id)) if pd.notna(category_id) else None)
    for tenant_id, category_id in zip(tickets["tenant_id"], tickets["maintenance_category_id"])
]

preventive_completed = executions[
    executions["completed_at"].notna() & executions["status"].eq("completed")
].copy()

empty_tickets = tickets.iloc[0:0]
empty_preventive = preventive_completed.iloc[0:0]

tickets_by_asset = {
    key: group.sort_values("created_at")
    for key, group in tickets.groupby(["tenant_id", "asset_card_id"])
}
corrective_by_asset = {
    key: group.sort_values("created_at")
    for key, group in tickets[tickets["is_corrective"]].groupby(["tenant_id", "asset_card_id"])
}
preventive_by_asset = {
    key: group.sort_values("completed_at")
    for key, group in preventive_completed.groupby(["tenant_id", "asset_card_id"])
}

print(f"Tickets correctivos: {int(tickets['is_corrective'].sum()):,}")
print(f"Ejecuciones preventivas completadas: {len(preventive_completed):,}")
tickets[["tenant_id", "id", "asset_card_id", "created_at", "maintenance_type", "priority", "status"]].head()


/tmp/ipykernel_2329/9506944.py:4: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  & (tickets["created_at"] <= HISTORY_END + pd.Timedelta(days=TARGET_WINDOW_DAYS))


Tickets correctivos: 4,179
Ejecuciones preventivas completadas: 8,274


,tenant_id,id,asset_card_id,created_at,maintenance_type,priority,status
0,acero-norte,1,1,2023-02-26 17:53:07,corrective,medium,closed
1,acero-norte,2,1,2023-03-14 03:00:03,corrective,critical,closed
2,acero-norte,3,1,2023-09-23 21:05:48,corrective,critical,closed
3,acero-norte,4,1,2023-11-12 11:54:01,corrective,high,closed
4,acero-norte,5,1,2023-12-24 10:57:18,corrective,medium,closed


## 4. Construcción de las 12,000 observaciones

24 cortes mensuales × 125 activos × 4 empresas. La ventana de predictores es de 180 días y el objetivo mira los siguientes 30 días.

In [5]:

rows = []

for asset in assets.itertuples(index=False):
    key = (asset.tenant_id, int(asset.asset_id))
    asset_tickets = tickets_by_asset.get(key, empty_tickets)
    corrective = corrective_by_asset.get(key, empty_tickets)
    preventive = preventive_by_asset.get(key, empty_preventive)

    failure_dates = pd.to_datetime(corrective["created_at"], errors="coerce").dropna().sort_values()
    closure_dates = pd.to_datetime(asset_tickets["closed_at"], errors="coerce").dropna()
    preventive_dates = pd.to_datetime(preventive["completed_at"], errors="coerce").dropna()

    for cutoff in CUTOFFS:
        feature_start = cutoff - pd.Timedelta(days=FEATURE_WINDOW_DAYS)
        target_end = cutoff + pd.Timedelta(days=TARGET_WINDOW_DAYS)

        prior = corrective[corrective["created_at"] < cutoff]
        recent_90 = prior[prior["created_at"] >= cutoff - pd.Timedelta(days=90)]
        recent_180 = prior[prior["created_at"] >= feature_start]

        last_failure = failure_dates[failure_dates < cutoff].max()
        days_since_failure = (
            (cutoff - last_failure).days if pd.notna(last_failure) else np.nan
        )

        if len(failure_dates[failure_dates < cutoff]) >= 2:
            diffs = failure_dates[failure_dates < cutoff].diff().dropna().dt.total_seconds() / 86400
            mtbf_days = diffs.mean()
        else:
            mtbf_days = np.nan

        last_preventive = preventive_dates[preventive_dates < cutoff].max()
        last_closure = closure_dates[closure_dates < cutoff].max()

        maintenance_candidates = [
            (last_preventive, "preventivo"),
            (last_closure, "correctivo"),
        ]
        maintenance_candidates = [
            (date, kind) for date, kind in maintenance_candidates if pd.notna(date)
        ]

        if maintenance_candidates:
            last_maintenance, last_maintenance_type = max(
                maintenance_candidates, key=lambda item: item[0]
            )
            days_since_maintenance = (cutoff - last_maintenance).days
        else:
            last_maintenance_type = None
            days_since_maintenance = np.nan

        priority_values = recent_180["priority_num"].dropna()
        avg_priority = priority_values.mean() if len(priority_values) else np.nan

        resolution_values = recent_180[
            recent_180["closed_at"].notna() & (recent_180["closed_at"] < cutoff)
        ]["resolution_hours"].dropna()
        avg_resolution = resolution_values.mean() if len(resolution_values) else np.nan

        last_category = (
            corrective[corrective["created_at"] < cutoff].sort_values("created_at")["categoria"].tail(1)
        )
        last_category_name = last_category.iloc[0] if len(last_category) else None

        has_target = (
            (corrective["created_at"] > cutoff)
            & (corrective["created_at"] <= target_end)
        ).any()

        rows.append(
            {
                "tenant_id": asset.tenant_id,
                "asset_id": int(asset.asset_id),
                "codigo_activo": asset.codigo_activo,
                "fecha_corte": cutoff,
                "ventana_inicio": feature_start,
                "ventana_fin": cutoff,
                "falla_30d": int(has_target),
                "antiguedad_dias": max(0, (cutoff - asset.installed_at).days),
                "tipo_activo": asset.tipo_activo,
                "sede": asset.sede,
                "criticidad": asset.criticidad,
                "fallas_90d": int(len(recent_90)),
                "fallas_180d": int(len(recent_180)),
                "dias_desde_ultimo_mantenimiento": days_since_maintenance,
                "tipo_ultimo_mantenimiento": last_maintenance_type,
                "dias_desde_ultima_falla": days_since_failure,
                "mtbf_dias": mtbf_days,
                "prioridad_promedio": avg_priority,
                "resolucion_promedio_h": avg_resolution,
                "causa_ultima_falla": last_category_name,
            }
        )

dataset = pd.DataFrame(rows).sort_values(
    ["tenant_id", "asset_id", "fecha_corte"]
).reset_index(drop=True)
dataset["falla_30d"] = dataset["falla_30d"].astype(int)

print(f"Dataset: {dataset.shape[0]:,} filas x {dataset.shape[1]} columnas")
print(f"Tasa de positivos: {dataset['falla_30d'].mean():.2%}")
dataset.head().to_string()


/tmp/ipykernel_2329/1057638500.py:14: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  feature_start = cutoff - pd.Timedelta(days=FEATURE_WINDOW_DAYS)
/tmp/ipykernel_2329/1057638500.py:15: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_end = cutoff + pd.Timedelta(days=TARGET_WINDOW_DAYS)
/tmp/ipykernel_2329/1057638500.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  recent_90 = prior[prior["created_at"] >= cutoff - pd.Timedelta(days=90)]
/tmp/ipykernel_2329/1057638500.py:14: Dep

/tmp/ipykernel_2329/1057638500.py:14: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  feature_start = cutoff - pd.Timedelta(days=FEATURE_WINDOW_DAYS)
/tmp/ipykernel_2329/1057638500.py:15: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_end = cutoff + pd.Timedelta(days=TARGET_WINDOW_DAYS)
/tmp/ipykernel_2329/1057638500.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  recent_90 = prior[prior["created_at"] >= cutoff - pd.Timedelta(days=90)]
/tmp/ipykernel_2329/1057638500.py:14: Dep

/tmp/ipykernel_2329/1057638500.py:14: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  feature_start = cutoff - pd.Timedelta(days=FEATURE_WINDOW_DAYS)
/tmp/ipykernel_2329/1057638500.py:15: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_end = cutoff + pd.Timedelta(days=TARGET_WINDOW_DAYS)
/tmp/ipykernel_2329/1057638500.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  recent_90 = prior[prior["created_at"] >= cutoff - pd.Timedelta(days=90)]
/tmp/ipykernel_2329/1057638500.py:14: Dep

/tmp/ipykernel_2329/1057638500.py:14: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  feature_start = cutoff - pd.Timedelta(days=FEATURE_WINDOW_DAYS)
/tmp/ipykernel_2329/1057638500.py:15: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_end = cutoff + pd.Timedelta(days=TARGET_WINDOW_DAYS)
/tmp/ipykernel_2329/1057638500.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  recent_90 = prior[prior["created_at"] >= cutoff - pd.Timedelta(days=90)]
/tmp/ipykernel_2329/1057638500.py:14: Dep

/tmp/ipykernel_2329/1057638500.py:14: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  feature_start = cutoff - pd.Timedelta(days=FEATURE_WINDOW_DAYS)
/tmp/ipykernel_2329/1057638500.py:15: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_end = cutoff + pd.Timedelta(days=TARGET_WINDOW_DAYS)
/tmp/ipykernel_2329/1057638500.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  recent_90 = prior[prior["created_at"] >= cutoff - pd.Timedelta(days=90)]
/tmp/ipykernel_2329/1057638500.py:14: Dep

/tmp/ipykernel_2329/1057638500.py:14: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  feature_start = cutoff - pd.Timedelta(days=FEATURE_WINDOW_DAYS)
/tmp/ipykernel_2329/1057638500.py:15: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_end = cutoff + pd.Timedelta(days=TARGET_WINDOW_DAYS)
/tmp/ipykernel_2329/1057638500.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  recent_90 = prior[prior["created_at"] >= cutoff - pd.Timedelta(days=90)]
/tmp/ipykernel_2329/1057638500.py:14: Dep

... (212 outputs left out)

/tmp/ipykernel_2329/1057638500.py:14: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  feature_start = cutoff - pd.Timedelta(days=FEATURE_WINDOW_DAYS)
/tmp/ipykernel_2329/1057638500.py:15: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_end = cutoff + pd.Timedelta(days=TARGET_WINDOW_DAYS)
/tmp/ipykernel_2329/1057638500.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  recent_90 = prior[prior["created_at"] >= cutoff - pd.Timedelta(days=90)]
/tmp/ipykernel_2329/1057638500.py:14: Dep

/tmp/ipykernel_2329/1057638500.py:14: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  feature_start = cutoff - pd.Timedelta(days=FEATURE_WINDOW_DAYS)
/tmp/ipykernel_2329/1057638500.py:15: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_end = cutoff + pd.Timedelta(days=TARGET_WINDOW_DAYS)
/tmp/ipykernel_2329/1057638500.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  recent_90 = prior[prior["created_at"] >= cutoff - pd.Timedelta(days=90)]
/tmp/ipykernel_2329/1057638500.py:14: Dep

Dataset: 12,000 filas x 20 columnas
Tasa de positivos: 24.49%


'     tenant_id  asset_id codigo_activo fecha_corte ventana_inicio ventana_fin  falla_30d  antiguedad_dias tipo_activo          sede criticidad  fallas_90d  fallas_180d  dias_desde_ultimo_mantenimiento tipo_ultimo_mantenimiento  dias_desde_ultima_falla  mtbf_dias  prioridad_promedio  resolucion_promedio_h causa_ultima_falla\n0  acero-norte         1     DEMO-0001  2024-02-01     2023-08-05  2024-02-01          1             4023   Compresor  Planta norte   critical           3            4                               21                preventivo                     26.0  62.589613            3.250000                  28.50         Hidráulica\n1  acero-norte         1     DEMO-0001  2024-03-01     2023-09-03  2024-03-01          0             4052   Compresor  Planta norte   critical           3            5                               13                correctivo                     15.0  58.857429            3.000000                  30.60           Mecánica\n2  acero-norte       

## 5. Validación del dataset

In [6]:

expected_rows = len(assets) * len(CUTOFFS)
positive_rate = dataset["falla_30d"].mean()

assert len(dataset) == expected_rows, (
    f"Se esperaban {expected_rows} filas y se generaron {len(dataset)}"
)
assert dataset[["tenant_id", "asset_id", "fecha_corte"]].duplicated().sum() == 0
assert dataset["falla_30d"].isin([0, 1]).all()
assert dataset["fecha_corte"].between(CUTOFFS.min(), CUTOFFS.max()).all()

print(f"Validación de volumen: OK ({expected_rows:,} filas)")
print(f"Tasa de positivos: {positive_rate:.2%}")
if not 0.20 <= positive_rate <= 0.35:
    print("Aviso: la tasa de positivos está fuera del rango objetivo de 20%–35%.")
else:
    print("Tasa de positivos dentro del rango objetivo de 20%–35%.")

missing = dataset.isna().mean().sort_values(ascending=False)
print("\nPorcentaje de valores nulos por columna:")
print(missing[missing > 0].to_string())

print("\nDistribución por tenant:")
print(dataset.groupby("tenant_id")["falla_30d"].agg(["size", "mean"]).to_string())


Validación de volumen: OK (12,000 filas)
Tasa de positivos: 24.49%
Tasa de positivos dentro del rango objetivo de 20%–35%.

Porcentaje de valores nulos por columna:
resolucion_promedio_h      0.278167
prioridad_promedio         0.274750
mtbf_dias                  0.130667
dias_desde_ultima_falla    0.044000
causa_ultima_falla         0.044000

Distribución por tenant:
                    size      mean
tenant_id                         
acero-norte         3000  0.180667
alimentos-valle     3000  0.272333
logistica-pacifico  3000  0.333000
textiles-andinos    3000  0.193667


## 6. Persistencia y gráficos de control

In [7]:

dataset_path = PROCESSED / "predictive_maintenance_dataset.csv"
dataset.to_csv(dataset_path, index=False)

summary = {
    "rows": int(len(dataset)),
    "columns": int(dataset.shape[1]),
    "assets": int(dataset[["tenant_id", "asset_id"]].drop_duplicates().shape[0]),
    "tenants": int(dataset["tenant_id"].nunique()),
    "cutoffs": int(dataset["fecha_corte"].nunique()),
    "cutoff_min": str(dataset["fecha_corte"].min().date()),
    "cutoff_max": str(dataset["fecha_corte"].max().date()),
    "positive_rows": int(dataset["falla_30d"].sum()),
    "positive_rate": float(dataset["falla_30d"].mean()),
    "target": "falla_30d",
    "target_definition": "Falla correctiva en los 30 días posteriores al corte",
}

with (PROCESSED / "etl_summary.json").open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, ensure_ascii=False, indent=2)

missing.to_csv(PROCESSED / "missing_values.csv", header=["missing_rate"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
dataset.groupby("falla_30d").size().plot(
    kind="bar", ax=axes[0], color=["#64748b", "#dc2626"]
)
axes[0].set_title("Distribución del objetivo falla_30d")
axes[0].set_xlabel("falla_30d")
axes[0].set_ylabel("Filas")

dataset.groupby("fecha_corte")["falla_30d"].mean().plot(
    ax=axes[1], marker="o", color="#2563eb"
)
axes[1].set_title("Tasa de fallas a 30 días por corte")
axes[1].set_xlabel("Fecha de corte")
axes[1].set_ylabel("Tasa de positivos")
axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig(FIGURES / "dataset_target_distribution.png", dpi=150, bbox_inches="tight")
plt.close(fig)

print(f"Dataset guardado en: {dataset_path}")
print(json.dumps(summary, ensure_ascii=False, indent=2))


Dataset guardado en: /workspace/data/processed/predictive_maintenance_dataset.csv
{
  "rows": 12000,
  "columns": 20,
  "assets": 500,
  "tenants": 4,
  "cutoffs": 24,
  "cutoff_min": "2024-02-01",
  "cutoff_max": "2026-01-01",
  "positive_rows": 2939,
  "positive_rate": 0.24491666666666667,
  "target": "falla_30d",
  "target_definition": "Falla correctiva en los 30 días posteriores al corte"
}


El dataset está listo para `02_model_training_predictive_maintenance.ipynb`. La tasa de positivos y la distribución por corte quedan registradas en `data/processed/etl_summary.json` y `reports/figures/dataset_target_distribution.png`.